# Guide Capture：全量覆盖度与第一阶段细胞质控

从完整 Cell Ranger RNA/guide 结果出发，保留全量细胞分母、原始计数和来源信息。第一阶段不包含双细胞模型、能量距离、差异表达或 DNA 编辑率分析。

使用 `scripts/run_analysis_notebook.py --config <本地配置.json>` 执行。配置中的输入必须先通过交付方哈希及复制校验；运行结果和执行后的 notebook 保存在配置指定的数据盘目录，不写回代码仓库。

In [ ]:
import os, sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'scripts'))
from analyze_guide_capture_qc import run_analysis
config_path = Path(os.environ['PERTURBSEQ_ANALYSIS_CONFIG'])
config = json.loads(config_path.read_text(encoding='utf-8'))
output = Path(config['output_root'])
print('输出目录：', output)

## 全量测量、guide 对照、覆盖度、QC 和稀疏 h5ad

分块读取原始 RNA 计数，单 guide、multi guide 和无可靠分配全部保留。RNA QC 使用宽松的分布阈值：log1p 计数/feature 数下侧 5 MAD，辅以最低 500 UMI/200 feature；线粒体比例同时超过 3 MAD 和 8% 才排除。高计数仅标记诊断，待后续双细胞诊断。

每个输出 h5ad 重新打开，分块核对每个细胞的原始计数总和；文件 SHA-256 和代码版本记录在 provenance。

In [ ]:
summary = run_analysis(config_path)
{k: v for k, v in summary.items() if k not in ['outputs', 'figures', 'thresholds']}

In [ ]:
from IPython.display import display, Image, Markdown
for path in summary['figures']:
    display(Image(filename=path))
display(Markdown((output / 'reports/01_coverage_and_stage1_qc.md').read_text(encoding='utf-8')))